// 1. Import necessary libraries
IMPORT pandas, numpy
IMPORT matplotlib.pyplot, seaborn

// 2. Load and Initial Exploration
LOAD 'customer_segmentation.csv' into dataframe 'df'
PRINT df.head(), df.columns, df.shape   // 2240 rows, 29 columns
PRINT df.info()                          // check dtypes
CHECK df.isna().sum()                    // find missing values (24 total)

// 3. Data Cleaning
DROP rows with NA values, IN PLACE (df.dropna(inplace=True))
PRINT df.describe()                      // summary stats for numeric columns
PRINT value_counts() for 'Education' and 'Marital_Status'

// 4. Feature Engineering
CONVERT 'Dt_Customer' column to datetime (dayfirst=True)
CREATE 'Age' = 2025 - df['Year_Birth']
CREATE 'Total_Children' = df['Kidhome'] + df['Teenhome']
DEFINE spending_columns = ['MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds']
CREATE 'Total_Spending' = sum(df[spending_columns], axis=1)
CREATE 'Customer_Since' = (today's date - df['Dt_Customer']).days

// 5. Exploratory Data Analysis (EDA)
PLOT histogram (with KDE) of Age, bins=30 -> "Age Distribution"
PLOT histogram (with KDE) of Income, bins=30 -> "Income Distribution"
PLOT histogram (with KDE) of Total_Spending, bins=30 -> "Total Spending Distribution"

PLOT boxplot: X=Education, Y=Income -> "Income by Education Level"
PLOT boxplot: X=Marital_Status, Y=Total_Spending -> "Spending by Marital Status"

// 6. Correlation Analysis
SELECT columns: Income, Age, Total_Spending, NumWebPurchases, NumStorePurchases
COMPUTE correlation matrix on selected columns
PLOT heatmap of correlation matrix (cmap='coolwarm', annot=True) -> "Correlation Matrix"

// 7. Pivot Table & Grouped Analysis
CREATE pivot_table: values=Income, index=Education, columns=Marital_Status, aggfunc=mean
PLOT heatmap of pivot table (annot=True, fmt='.0f', cmap='YlGnBu') -> "Average Income by Education and Marital Status"

GROUP BY Education, aggregate Total_Spending mean, sort descending
PLOT as bar chart -> "Average Spending by Education"

// 8. Campaign Acceptance Analysis
CREATE 'Accepted_Any' = SUM(AcceptedCmp1..AcceptedCmp5, Response), axis=1
CONVERT 'Accepted_Any' to binary: 1 if > 0 else 0

GROUP BY Marital_Status, aggregate Accepted_Any mean, sort descending
PLOT as bar chart -> "Campaign Acceptance Rate by Marital Status"

// 9. Age Group Analysis
DEFINE bins = [18, 30, 40, 50, 60, 70, 90]
DEFINE labels = ['18-29','30-39','40-49','50-59','60-69','70+']
CREATE 'Age_Group' = pandas.cut(df['Age'], bins=bins, labels=labels)

GROUP BY Age_Group, aggregate Income mean
PLOT as horizontal bar chart -> "Average Income by Age Group"

// 10. Feature Selection for Clustering
DEFINE features = ['Age', 'Income', 'Total_Spending', 'NumWebPurchases',
                    'NumStorePurchases', 'NumWebVisitsMonth']
DEFINE X = df[features].copy()

// 11. Feature Scaling
IMPORT StandardScaler FROM sklearn.preprocessing
INITIALIZE scaler = StandardScaler()
COMPUTE X_scaled = scaler.fit_transform(X)

// 12. Determine Optimal Number of Clusters (Elbow Method)
IMPORT KMeans FROM sklearn.cluster
INITIALIZE empty list inertia_list
FOR k IN range(2, 11) DO:
    MODEL = KMeans(n_clusters=k)
    MODEL.fit(X_scaled)
    APPEND MODEL.inertia_ to inertia_list
PLOT range(2,11) vs inertia_list, with markers -> "Elbow Method for Optimal K"
VISUALLY SELECT k where curve bends (chosen: k = 6)

// 13. Train Final KMeans Model
INITIALIZE kmeans = KMeans(n_clusters=6)
CREATE df['Cluster'] = kmeans.fit_predict(X_scaled)

// 14. Analyze Cluster Characteristics
COMPUTE cluster_summary = df.groupby('Cluster')[features].mean()
PRINT cluster_summary
COMPUTE cluster value_counts (size of each cluster)
// Manually interpret and label clusters, e.g.:
//   "Premium customers" (high income, high spending)
//   "Digital buyers" (high web purchases, low store purchases)
//   "Dormant customers" (low recency/inactive)
//   "Budget customers" (low income, low spending)

// 15. Visualize Clusters with PCA
IMPORT PCA FROM sklearn.decomposition
INITIALIZE pca = PCA(n_components=2)
COMPUTE pca_data = pca.fit_transform(X_scaled)
CREATE df['PCA1'] = pca_data[:,0]
CREATE df['PCA2'] = pca_data[:,1]
PLOT scatterplot: X=PCA1, Y=PCA2, hue=Cluster, palette='Set1' -> "Customer Segmentation (PCA)"

// 16. Export Model and Scaler
IMPORT joblib
joblib.dump(kmeans, 'kmeans_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

// ============================================
// PART 2: Streamlit Web App (segmentation.py)
// ============================================

// 17. App Setup
IMPORT streamlit as st, pandas, numpy, joblib
LOAD kmeans = joblib.load('kmeans_model.pkl')
LOAD scaler = joblib.load('scaler.pkl')
SET st.title("Customer Segmentation App")
SET st.write("Enter customer details to predict the segment.")

// 18. Create Input Fields (matching training features)
age = st.number_input("Age", min=18, max=100, default=35)
income = st.number_input("Income", min=0, max=200000, default=50000)
total_spending = st.number_input("Total Spending", min=0, max=5000, default=1000)
num_web_purchases = st.number_input("Number of Web Purchases", min=0, max=100, default=10)
num_store_purchases = st.number_input("Number of Store Purchases", min=0, max=100, default=10)
num_web_visits = st.number_input("Number of Web Visits per Month", min=0, max=50, default=3)
// NOTE: recency was mentioned but not used in final feature set - optional field

// 19. Build Input DataFrame
CREATE input_data = pandas.DataFrame with columns matching 'features' list,
    using the values collected above (SAME ORDER as training features)

// 20. Scale Input & Predict
COMPUTE input_scaled = scaler.transform(input_data)

IF st.button("Predict Segment"):
    cluster = kmeans.predict(input_scaled)
    DISPLAY st.success(f"Predicted segment is Cluster {cluster}")
    // Optionally map cluster number -> human-readable label here

// 21. Run App
// Command line: streamlit run segmentation.py

In [ ]:
import kagglehub
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from datetime import date
import seaborn as sns


path = kagglehub.dataset_download("vishakhdapat/customer-segmentation-clustering")

print("Path to dataset files:", path)

In [ ]:
'''
// 2. Load and Initial Exploration
LOAD 'customer_segmentation.csv' into dataframe 'df'
PRINT df.head(), df.columns, df.shape   // 2240 rows, 29 columns
PRINT df.info()                          // check dtypes
CHECK df.isna().sum()                    // find missing values (24 total)
'''
df = pd.read_csv(path + '/customer_segmentation.csv')
print("head: ", df.head())
print("cols: ", df.columns)
print("Shape: ", df.shape)
print("info: ", df.info())
print("missing vals: ", df.isna().sum())


We see that the income is missing 24 rows
I think i need to 
1. make dummy vars for the str cols.
2. erase the Nan rows
3. not sure if we should Delete ID
4. somehow normalize DOB? its a weird string 
5. income is a float when eveyrthing is a string, should we be worried? Were gonnna sacale it so no worries

In [ ]:
'''
// 3. Data Cleaning
DROP rows with NA values, IN PLACE (df.dropna(inplace=True))
PRINT df.describe()                      // summary stats for numeric columns
PRINT value_counts() for 'Education' and 'Marital_Status'
'''
df.dropna(inplace=True)
print(df.describe())
print("value counts: ", df['Education'].value_counts, df['Marital_Status'].value_counts)

This is data cleaning by dropping the Nan rows

Some of these floats could be truncated and some of these columns dont need these stats such as ID. 

In [ ]:
'''

// 4. Feature Engineering
CONVERT 'Dt_Customer' column to datetime (dayfirst=True)
CREATE 'Age' = 2025 - df['Year_Birth']
CREATE 'Total_Children' = df['Kidhome'] + df['Teenhome']
DEFINE spending_columns = ['MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds']
CREATE 'Total_Spending' = sum(df[spending_columns], axis=1)
CREATE 'Customer_Since' = (today's date - df['Dt_Customer']).days

'''

today = date.today()
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], dayfirst=True) 

df["Age"] = 2025 - df['Year_Birth']
df["Total_Children"] = df['Kidhome'] + df['Teenhome']
spending_columns = ['MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds']
df["Total_Spending"] = df[spending_columns].sum(axis=1)
df["Customer_Since"] = (pd.Timestamp.today() - df['Dt_Customer']).dt.days


we are creating new features from the data we have in order to figure out deeper relationships within the data than the current columns we have.

In [ ]:
'''
// 5. Exploratory Data Analysis (EDA)
PLOT histogram (with KDE) of Age, bins=30 -> "Age Distribution"
PLOT histogram (with KDE) of Income, bins=30 -> "Income Distribution"
PLOT histogram (with KDE) of Total_Spending, bins=30 -> "Total Spending Distribution"

PLOT boxplot: X=Education, Y=Income -> "Income by Education Level"
PLOT boxplot: X=Marital_Status, Y=Total_Spending -> "Spending by Marital Status"
'''
columns_to_plot = ['Age', 'Income', 'Total_Spending']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, columns_to_plot):
    sns.histplot(df[col], bins=30, kde=True, ax=ax)
    ax.set_title(f"{col} Distribution")

plt.tight_layout()
plt.show()

# BOXPLOTTTTT MISSING       
sns.boxplot(data=df, )


In [ ]:
'''
// 6. Correlation Analysis
SELECT columns: Income, Age, Total_Spending, NumWebPurchases, NumStorePurchases
COMPUTE correlation matrix on selected columns
PLOT heatmap of correlation matrix (cmap='coolwarm', annot=True) -> "Correlation Matrix"
'''

corr_cols = ['Income', 'Age', 'Total_Spending', 'NumWebPurchases', 'NumStorePurchases']
corr_matrix = df[corr_cols].corr()
sns.heatmap(corr_matrix, cmap='coolwarm', annot=True)
plt.title('Correlation Matrix')
plt.show()


We selected the desired columns to see the correlation between each of the columns with the others to better unerstand the relationship hidden in the data.  

In [ ]:
'''
// 7. Pivot Table & Grouped Analysis
CREATE pivot_table: values=Income, index=Education, columns=Marital_Status, aggfunc=mean
PLOT heatmap of pivot table (annot=True, fmt='.0f', cmap='YlGnBu') -> "Average Income by Education and Marital Status"
'''
pivot = df.pivot_table(values='Income', index='Education', columns='Marital_Status', aggfunc='mean')
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGnBu')
plt.title('Average Income by Education and Marital Status')
plt.show()

we created a pivot table adn heat map to illustrate the relationship between education, marital status, and income. The axes are marital status and education to show a 1 to 1 relatoinship between their columns and the income is the heatmap because it better illustrates where the higher and lower incomes are. 

In [ ]:
'''
// 8. Campaign Acceptance Analysis
CREATE 'Accepted_Any' = SUM(AcceptedCmp1..AcceptedCmp5, Response), axis=1
CONVERT 'Accepted_Any' to binary: 1 if > 0 else 0

GROUP BY Marital_Status, aggregate Accepted_Any mean, sort descending
PLOT as bar chart -> "Campaign Acceptance Rate by Marital Status"
'''
accept_any = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response']
df['Accepted_Any'] = df[accept_any].sum(axis=1)
df['Accepted_Any']= (df['Accepted_Any']>= 1).astype(int)

grouped = df.groupby(['Marital_Status'])
grouped = grouped['Accepted_Any'].mean().sort_values(ascending=False)

sns.barplot(data=df, x='Marital_Status', y='Accepted_Any', order=grouped.index)
plt.title("Campaign Acceptance Rate by Marital Status")
plt.show()

We wanted to see the relationship between marital status and campaign acceptance. We feature engineered the accepted_any column and plotted it against the marital statuses.
write down what we learned for box

In [ ]:
'''
// 9. Age Group Analysis
DEFINE bins = [18, 30, 40, 50, 60, 70, 90]
DEFINE labels = ['18-29','30-39','40-49','50-59','60-69','70+']
CREATE 'Age_Group' = pandas.cut(df['Age'], bins=bins, labels=labels)

GROUP BY Age_Group, aggregate Income mean
PLOT as horizontal bar chart -> "Average Income by Age Group"
'''
bins = [18, 30, 40, 50, 60, 70, 90]
labels = ['18-29','30-39','40-49','50-59','60-69','70+']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels)

grouped = df.groupby(['Age_Group'])
grouped = grouped['Income'].mean()

sns.barplot(data=df, x='Income', y='Age_Group', order=grouped.index, orient="y")
plt.title("Average Income by Age Group")
plt.show()



what is pd.cut: just bins integers and u can give the bins names
We wanted to plot the relationship between age groups and income. We binned the ages inot these groups and then plotted their incomes on a bar chart. 

In [ ]:
'''
// 10. Feature Selection for Clustering
DEFINE features = ['Age', 'Income', 'Total_Spending', 'NumWebPurchases',
                    'NumStorePurchases', 'NumWebVisitsMonth']
DEFINE X = df[features].copy()
'''
features = ['Age', 'Income', 'Total_Spending', 'NumWebPurchases','NumStorePurchases', 'NumWebVisitsMonth']
X = df[features].copy()


what does .copy do that just assigning it cant?
We want to select the features of the data set that we want to cluster 

In [ ]:

'''
// 11. Feature Scaling
IMPORT StandardScaler FROM sklearn.preprocessing
INITIALIZE scaler = StandardScaler()
COMPUTE X_scaled = scaler.fit_transform(X)
'''
from sklearn.preprocessing import StandardScaler 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

We are scaling the data that we want to cluster to train our model

In [ ]:
'''// 12. Determine Optimal Number of Clusters (Elbow Method)
IMPORT KMeans FROM sklearn.cluster
INITIALIZE empty list inertia_list
FOR k IN range(2, 11) DO:
    MODEL = KMeans(n_clusters=k)
    MODEL.fit(X_scaled)
    APPEND MODEL.inertia_ to inertia_list
PLOT range(2,11) vs inertia_list, with markers -> "Elbow Method for Optimal K"
VISUALLY SELECT k where curve bends (chosen: k = 6)
'''
from sklearn.cluster import KMeans
inertia_list = []
for k in range(2, 11):
    model = KMeans(n_clusters=k)
    model.fit(X_scaled)
    inertia_list.append(model.inertia_)

plt.plot(range(2,11), inertia_list)
plt.title("Elbow Method for Optimal K")
plt.show()


elbow method is visually choosing k where the WCSS stops dropping off rapidly. We can see the k=6 is the elbow and therefore the optimal k

In [ ]:
'''
// 13. Train Final KMeans Model
INITIALIZE kmeans = KMeans(n_clusters=6)
CREATE df['Cluster'] = kmeans.fit_predict(X_scaled)
'''
kmeans = KMeans(n_clusters=6)
df['Cluster'] = kmeans.fit_predict(X_scaled)



In [ ]:
'''
// 14. Analyze Cluster Characteristics
COMPUTE cluster_summary = df.groupby('Cluster')[features].mean()
PRINT cluster_summary
COMPUTE cluster value_counts (size of each cluster)
// Manually interpret and label clusters, e.g.:
//   "Premium customers" (high income, high spending)
//   "Digital buyers" (high web purchases, low store purchases)
//   "Dormant customers" (low recency/inactive)
//   "Budget customers" (low income, low spending)
'''
cluster_summary = df.groupby("Cluster")[features].mean()
print(f"Cluster Summary: {cluster_summary}")
print("value counts: ", df['Cluster'].value_counts)

In [ ]:
'''// 15. Visualize Clusters with PCA
IMPORT PCA FROM sklearn.decomposition
INITIALIZE pca = PCA(n_components=2)
COMPUTE pca_data = pca.fit_transform(X_scaled)
CREATE df['PCA1'] = pca_data[:,0]
CREATE df['PCA2'] = pca_data[:,1]
PLOT scatterplot: X=PCA1, Y=PCA2, hue=Cluster, palette='Set1' -> "Customer Segmentation (PCA)"
'''
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
pca_data = pca.fit_transform(X_scaled)
df['PCA1'] = pca_data[:,0]
df['PCA2'] = pca_data[:,1]
sns.scatterplot(data=df, X=df['PCA1'], Y=df['PCA2'], hue=cluster_summary, palette='Set1')
plt.title("Customer Segmentation (PCA)")
plt.show()

In [ ]:
'''// 16. Export Model and Scaler
IMPORT joblib
joblib.dump(kmeans, 'kmeans_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
'''
import joblib
joblib.dump(kmeans, 'kmeans_model.pkl')
joblib.dump(scaler, 'scaler.pkl')